In [1]:
# Data manipulation
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# F1 data
import fastf1
from fastf1 import plotting

# Machine Learning
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import accuracy_score, mean_absolute_error, classification_report
from sklearn.preprocessing import LabelEncoder
import xgboost as xgb

# Utilities
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# Configure plotting
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# Enable FastF1 cache to speed up data loading
fastf1.Cache.enable_cache("cache")

print("Libraries imported successfully!")
race = fastf1.get_session(2024, 'Abu Dhabi Grand Prix', 'R')
race.load()  # This fetches the data

def get_qualifying_results(year, event_name):

    try:
        # Load qualifying session
        quali = fastf1.get_session(year, event_name, 'Q')
        quali.load()
        
        # Get qualifying results
        results = quali.results
        
        # Select relevant columns
        quali_data = results[[
            'Position', 'Abbreviation', 'TeamName', 'Q1', 'Q2', 'Q3'
        ]].copy()
        
        # Sort by position
        quali_data = quali_data.sort_values('Position')
        
        return quali_data
    
    except Exception as e:
        print(f"Error loading data for {year} {event_name}: {e}")
        return None

# Fetch Abu Dhabi GP qualifying results
print("="*80)
print("ABU DHABI GRAND PRIX - QUALIFYING RESULTS")
print("="*80)

# 2023 Abu Dhabi GP
print("\n🏁 2023 ABU DHABI GRAND PRIX - QUALIFYING\n")
quali_2023 = get_qualifying_results(2023, 'Abu Dhabi Grand Prix')
if quali_2023 is not None:
    print(quali_2023.to_string(index=False))

print("\n" + "="*80)

# 2024 Abu Dhabi GP
print("\n🏁 2024 ABU DHABI GRAND PRIX - QUALIFYING\n")
quali_2024 = get_qualifying_results(2024, 'Abu Dhabi Grand Prix')
if quali_2024 is not None:
    print(quali_2024.to_string(index=False))

print("\n" + "="*80)

# 2025 Abu Dhabi GP
print("\n🏁 2025 ABU DHABI GRAND PRIX - QUALIFYING\n")
quali_2025_results = get_qualifying_results(2025, 'Abu Dhabi Grand Prix')
if quali_2025_results is not None:
    print(quali_2025_results.to_string(index=False))

print("\n" + "="*80)

def get_race_results(year, event_name):
    try:
        # Load qualifying session
        race = fastf1.get_session(year, event_name, 'R')
        race.load()
        
        # Get qualifying results
        results = race.results.copy()
        
        # Select relevant columns
        race_data = results[[
            'Position', 'Abbreviation', 'TeamName', 'Time', 'GridPosition'
        ]].copy()
        
        # Sort by position
        race_data = race_data.sort_values('Position')
        
        return race_data
    
    except Exception as e:
        print(f"Error loading data for {year} {event_name}: {e}")
        return None
    
    # Fetch Abu Dhabi GP race results
print("="*80)
print("ABU DHABI GRAND PRIX - RACE RESULTS")
print("="*80)

# 2023 Abu Dhabi GP
print("\n🏁 2023 ABU DHABI GRAND PRIX - RACE\n")
race_2023 = get_race_results(2023, 'Abu Dhabi Grand Prix')
if race_2023 is not None:
    print(race_2023.to_string(index=False))

print("\n" + "="*80)

# 2024 Abu Dhabi GP
print("\n🏁 2024 ABU DHABI GRAND PRIX - RACE\n")
race_2024 = get_race_results(2024, 'Abu Dhabi Grand Prix')
if race_2024 is not None:
    print(race_2024.to_string(index=False))

print("\n" + "="*80)

# Load 2024 Abu Dhabi GP race session
session_2024 = fastf1.get_session(2024, "Abu Dhabi Grand Prix", "R")
session_2024.load()

# Extract lap and sector times
laps_2024 = session_2024.laps[["Driver", "LapTime", "Sector1Time", "Sector2Time", "Sector3Time"]].copy()
laps_2024.dropna(inplace=True)

# Convert times to seconds
for col in ["LapTime", "Sector1Time", "Sector2Time", "Sector3Time"]:
    laps_2024[f"{col} (s)"] = laps_2024[col].dt.total_seconds()

# Group by driver to get average sector times and lap times per driver
sector_times_2024 = laps_2024.groupby("Driver")[["Sector1Time (s)", "Sector2Time (s)", "Sector3Time (s)"]].mean().reset_index()
avg_lap_times_2024 = laps_2024.groupby("Driver")["LapTime (s)"].mean().reset_index()

# Calculate average lap time from 2024 data for use as fallback
avg_lap_time_2024 = laps_2024["LapTime (s)"].mean()

# Try to fetch 2025 Qualifying Data from Abu Dhabi GP
try:
    quali_session_2025 = fastf1.get_session(2025, 'Abu Dhabi Grand Prix', 'Q')
    quali_session_2025.load()
    
    # Extract qualifying times and driver info
    quali_results_2025 = quali_session_2025.results.copy()
    
    # Get the best qualifying time for each driver (Q3, Q2, or Q1)
    def get_best_quali_time(row):
        for col in ['Q3', 'Q2', 'Q1']:
            if pd.notna(row[col]):
                return row[col].total_seconds()
        return None
    
    qualifying_2025 = pd.DataFrame({
        'Driver': quali_results_2025['Abbreviation'].values,
        'DriverFullName': (quali_results_2025['FirstName'].astype(str) + ' ' + quali_results_2025['LastName'].astype(str)).values,
        'QualifyingTime (s)': quali_results_2025.apply(get_best_quali_time, axis=1).values
    })
    
    qualifying_2025 = qualifying_2025.dropna(subset=['QualifyingTime (s)'])
    print("✓ 2025 Qualifying data loaded successfully")
    
except Exception as e:
    print(f"⚠ 2025 Qualifying data not available: {e}")
    print(f"Using average lap time ({avg_lap_time_2024:.2f}s) as fallback")
    
    # Use average lap times from 2024 as qualifying times
    qualifying_2025 = pd.DataFrame({
        'Driver': sector_times_2024['Driver'].values,
        'DriverFullName': sector_times_2024['Driver'].values,
        'QualifyingTime (s)': avg_lap_time_2024
    })

# Merge with sector times from 2024
merged_data = qualifying_2025.merge(sector_times_2024, left_on="Driver", right_on="Driver", how="inner")

# Merge with average lap times from 2024 for training labels
merged_data = merged_data.merge(avg_lap_times_2024, left_on="Driver", right_on="Driver", how="inner")

# Now extract X and y from the same merged dataframe to ensure alignment
X = merged_data[["QualifyingTime (s)", "Sector1Time (s)", "Sector2Time (s)", "Sector3Time (s)"]].reset_index(drop=True)
y = merged_data["LapTime (s)"].reset_index(drop=True)

print(f"\nDataset info: X has {len(X)} samples, y has {len(y)} samples")
print(f"Drivers in dataset: {len(merged_data)}")

# Train Gradient Boosting Model
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=38)
model = GradientBoostingRegressor(n_estimators=200, learning_rate=0.1, random_state=38)
model.fit(X_train, y_train)

# Predict race times using 2025 qualifying and sector data
predicted_race_times = model.predict(X)
merged_data["PredictedRaceTime (s)"] = predicted_race_times

# Rank drivers by predicted race time
merged_data = merged_data.sort_values(by="PredictedRaceTime (s)")

# Print final predictions
print("\n🏁 Predicted 2025 Abu Dhabi GP Winner with Sector Times 🏁\n")
print(merged_data[["DriverFullName", "PredictedRaceTime (s)"]].to_string(index=False))

# Evaluate Model
y_pred = model.predict(X_test)
print(f"\n🔍 Model Error (MAE): {mean_absolute_error(y_test, y_pred):.2f} seconds")

core           INFO 	Loading data for Abu Dhabi Grand Prix - Race [v3.6.1]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...


Libraries imported successfully!


req            INFO 	Using cached data for car_data
req            INFO 	Using cached data for position_data
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 20 drivers: ['4', '55', '16', '44', '63', '1', '10', '27', '14', '81', '23', '22', '24', '18', '61', '20', '30', '77', '43', '11']
core           INFO 	Loading data for Abu Dhabi Grand Prix - Qualifying [v3.6.1]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...


ABU DHABI GRAND PRIX - QUALIFYING RESULTS

🏁 2023 ABU DHABI GRAND PRIX - QUALIFYING



req            INFO 	Using cached data for car_data
req            INFO 	Using cached data for position_data
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 20 drivers: ['1', '16', '81', '63', '4', '22', '14', '27', '11', '10', '44', '31', '18', '23', '3', '55', '20', '77', '24', '2']
core           INFO 	Loading data for Abu Dhabi Grand Prix - Qualifying [v3.6.1]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...


...

 Position Abbreviation        TeamName                   Time  GridPosition
      1.0          VER Red Bull Racing 0 days 01:27:02.624000           1.0
      2.0          LEC         Ferrari 0 days 00:00:17.993000           2.0
      3.0          RUS        Mercedes 0 days 00:00:20.328000           4.0
      4.0          PER Red Bull Racing 0 days 00:00:21.453000           9.0
      5.0          NOR         McLaren 0 days 00:00:24.284000           5.0
      6.0          PIA         McLaren 0 days 00:00:31.487000           3.0
      7.0          ALO    Aston Martin 0 days 00:00:39.512000           7.0
      8.0          TSU      AlphaTauri 0 days 00:00:43.088000           6.0
      9.0          HAM        Mercedes 0 days 00:00:44.424000          11.0
     10.0          STR    Aston Martin 0 days 00:00:55.632000          13.0
     11.0          RIC      AlphaTauri 0 days 00:00:56.229000          15.0
     12.0          OCO          Alpine 0 days 00:01:06.373000          12.0
     13.0   

req            INFO 	Using cached data for car_data
req            INFO 	Using cached data for position_data
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 20 drivers: ['4', '55', '16', '44', '63', '1', '10', '27', '14', '81', '23', '22', '24', '18', '61', '20', '30', '77', '43', '11']
core           INFO 	Loading data for Abu Dhabi Grand Prix - Race [v3.6.1]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...


 Position Abbreviation        TeamName                   Time  GridPosition
      1.0          NOR         McLaren 0 days 01:26:33.291000           1.0
      2.0          SAI         Ferrari 0 days 00:00:05.832000           3.0
      3.0          LEC         Ferrari 0 days 00:00:31.928000          19.0
      4.0          HAM        Mercedes 0 days 00:00:36.483000          16.0
      5.0          RUS        Mercedes 0 days 00:00:37.538000           6.0
      6.0          VER Red Bull Racing 0 days 00:00:49.847000           4.0
      7.0          GAS          Alpine 0 days 00:01:12.560000           5.0
      8.0          HUL    Haas F1 Team 0 days 00:01:15.554000           7.0
      9.0          ALO    Aston Martin 0 days 00:01:22.373000           8.0
     10.0          PIA         McLaren 0 days 00:01:23.821000           2.0
     11.0          ALB        Williams 0 days 00:00:11.251000          18.0
     12.0          TSU              RB 0 days 00:00:14.738000          11.0
     13.0   

req            INFO 	Using cached data for car_data
req            INFO 	Using cached data for position_data
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 20 drivers: ['4', '55', '16', '44', '63', '1', '10', '27', '14', '81', '23', '22', '24', '18', '61', '20', '30', '77', '43', '11']
core           INFO 	Loading data for Abu Dhabi Grand Prix - Qualifying [v3.6.1]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
req            INFO 	Using cached data for car_data
req            INFO 	Using cached data for position_d

✓ 2025 Qualifying data loaded successfully

Dataset info: X has 15 samples, y has 15 samples
Drivers in dataset: 15

🏁 Predicted 2025 Abu Dhabi GP Winner with Sector Times 🏁

  DriverFullName  PredictedRaceTime (s)
    Lando Norris              89.511386
    Carlos Sainz              89.581123
 Charles Leclerc              89.957456
  Lewis Hamilton              89.998070
  George Russell              90.107895
  Max Verstappen              90.239088
    Pierre Gasly              90.336937
 Nico Hulkenberg              90.661286
   Oscar Piastri              90.760614
 Fernando Alonso              90.868386
    Yuki Tsunoda              91.130116
 Alexander Albon              91.131286
    Lance Stroll              91.222143
     Liam Lawson              92.183926
Franco Colapinto              94.716960

🔍 Model Error (MAE): 0.18 seconds
